In [0]:
df = spark.read.csv("/Volumes/test-external-catalog/default/test-external-volume/Employee_Attrition.csv", header=True, inferSchema=True)
display(df)

In [0]:
# Filter high risk attrition employees
high_risk_df = df.filter((df.Attrition == "No") & (df.JobSatisfaction < 3))

# Select relevant columns
selected_df = high_risk_df.select("EmployeeNumber", "Attrition", "JobSatisfaction", "Department", "Age", "Gender")

# Write to Delta table in default schema of test-external-catalog
selected_df.write.format("delta").mode("overwrite").saveAsTable("`test-external-catalog`.default.high_risk_attrition_employees")

In [0]:
display(spark.sql("""
  DESCRIBE HISTORY `test-external-catalog`.default.high_risk_attrition_employees
"""))

In [0]:
high_risk_attrition_employees_df = spark.read.table("`test-external-catalog`.default.high_risk_attrition_employees")
display(high_risk_attrition_employees_df)

In [0]:
from pyspark.sql import Row

dummy_row = Row(EmployeeNumber=999999, Attrition="No", JobSatisfaction=1, Department="DummyDept", Age=30, Gender="DummyGender")
target_schema = spark.table("`test-external-catalog`.default.high_risk_attrition_employees").schema
dummy_df = spark.createDataFrame([dummy_row], schema=target_schema)

dummy_df.write.format("delta").mode("append").saveAsTable("`test-external-catalog`.default.high_risk_attrition_employees")

In [0]:
versions_df = spark.sql("""
  DESCRIBE HISTORY `test-external-catalog`.default.high_risk_attrition_employees
""")
display(versions_df.select(
    "version",
    "timestamp",
    "operation",
    "readVersion",
    "isBlindAppend",
    "operationMetrics"
).orderBy("version", ascending=False))

In [0]:
high_risk_attrition_employees_df = spark.read.table("`test-external-catalog`.default.high_risk_attrition_employees")
display(high_risk_attrition_employees_df)

In [0]:
high_risk_attrition_employees_df_v1 = spark.read.option("versionAsOf", 0).table("`test-external-catalog`.default.high_risk_attrition_employees")
display(high_risk_attrition_employees_df_v1)

In [0]:
spark.sql("""
  CREATE VOLUME IF NOT EXISTS `test-external-catalog`.default.employee_transformmed_data
""")

In [0]:
# Logical transformation: Add a column indicating if employee is at risk (JobSatisfaction < 3 and Attrition == "No")
transformed_df = df.withColumn(
    "AtRisk",
    ((df.JobSatisfaction < 3) & (df.Attrition == "No")).cast("boolean")
)

# Write to volume, partitioned by Department, in Parquet format
transformed_df.write.mode("overwrite") \
    .partitionBy("Department") \
    .parquet("/Volumes/test-external-catalog/default/employee_transformmed_data/")